# 08 · Data and model monitoring
**AAI-540 · Group 4 · Criteo CTR**

Three monitors, each answering a different question:

| Monitor | Question | Needs labels? |
|---|---|---|
| **Data quality** | is the traffic still the kind of traffic the model was trained on? | no |
| **Model quality** | is the model still right? | yes, once the attribution window closes |
| **Clarify (explainability)** | is the model still relying on the same features? | no |

**Why the order matters.** Clicks arrive minutes to hours after an impression,
so model quality is always the *last* signal to become available. By the time
log loss moves, the money has been spent. The data-quality monitor is
therefore the early warning, and for this dataset its most useful single
measure is not the generic numerical drift check — it is the **rate of unseen
categorical levels**, the share of rows whose advertiser or publisher the
encoder never saw during training. That needs no labels and rises the moment
the inventory mix changes.

**On thresholds.** The rare-bucket share is judged against the *training*
window, not an absolute number. High-cardinality Criteo columns already send a
large share of training rows to the rare bucket by construction (any level
below `RARE_MIN_COUNT`), so an absolute ceiling would fire on healthy traffic
and the monitor would be muted within a week. `monitoring.baseline_constraints`
records the training-time rate per column and
`monitoring.data_quality_report` alarms on the *increase*.

Everything in `src/criteo_ctr/monitoring.py` is unit-tested in
`tests/test_monitoring.py`, including the degradation cases, so the monitors
are known to fire for the right reasons before any live data reaches them.

In [ ]:
import os, sys, json, time, datetime as dt
sys.path.insert(0, os.path.abspath("../src"))

import boto3
import numpy as np
import pandas as pd
import sagemaker

from criteo_ctr import config as C
from criteo_ctr import evaluation as E
from criteo_ctr import features as F
from criteo_ctr import monitoring as M
from criteo_ctr.io_utils import Store, s3_uri, canonical_columns

sess = sagemaker.Session()
boto_sess = sess.boto_session
sm = boto_sess.client("sagemaker")
region = sess.boto_region_name
role = sagemaker.get_execution_role()
bucket = C.BUCKET or sess.default_bucket()
store = Store(bucket, boto_sess)
s3 = boto_sess.client("s3")

MON = f"{C.PREFIX}/monitoring"
ENDPOINT_NAME = "criteo-ctr-endpoint"
CAPTURE_PREFIX = f"{MON}/data-capture"
GROUND_TRUTH_PREFIX = f"{MON}/ground-truth"

FIG_DIR = os.path.abspath("../reports/figures"); os.makedirs(FIG_DIR, exist_ok=True)
deployment = json.loads(s3.get_object(
    Bucket=bucket, Key=f"{C.ARTIFACTS_PREFIX}/deployment_record.json")["Body"].read())
model_record = json.loads(s3.get_object(
    Bucket=bucket, Key=f"{C.ARTIFACTS_PREFIX}/model_record.json")["Body"].read())
baseline_metrics = model_record["evaluation"]["binary_classification_metrics"]
print(f"endpoint {deployment['endpoint_name']}")
print(f"training-time baseline: log loss {baseline_metrics['log_loss']['value']:.6f}, "
      f"AUC {baseline_metrics['auc']['value']:.4f}")

## 1 · Baselines from the training split

The baseline must describe the data the model was actually trained on, so it is
computed from the training split (and in the pipeline, inside the same job that
creates that split — see `scripts/preprocess.py`). A baseline computed from a
later sample would quietly bake the drift we are trying to detect into the
reference.

In [ ]:
splits = {name: canonical_columns(
              store.read_parquet_prefix(f"{C.SPLITS_PREFIX}/parquet/split={name}/"))
          for name in ("train", "production")}
train, production = splits["train"], splits["production"]
feature_cols = F.model_feature_columns()

statistics = M.baseline_statistics(train[feature_cols])
categorical_rates = M.unseen_level_report(train)["per_column"]
constraints = M.baseline_constraints(statistics, categorical_rates=categorical_rates)

store.put_json(statistics, f"{MON}/baseline/statistics.json")
store.put_json(constraints, f"{MON}/baseline/constraints.json")

print(f"baseline over {statistics['dataset']['item_count']:,} training rows, "
      f"{len(statistics['features'])} features")
print(f"training-time rare-bucket share: mean "
      f"{np.mean([v['rare_rate'] for v in categorical_rates.values()]):.2%}")
print("\nsanity check - the baseline must validate against its own data:")
self_check = M.data_quality_report(constraints, train[feature_cols], baseline_frame=train[feature_cols])
print(f"  {'PASS' if self_check['passed'] else 'FAIL'}: {len(self_check['violations'])} violations")

## 2 · Data-quality monitor: the production window against the baseline

Run our own report first, because it is the one we can explain line by line,
then hand the same baselines to the managed Model Monitor so the scheduled
version produces the standard artefacts.

In [ ]:
report = M.data_quality_report(constraints, production[feature_cols],
                               baseline_frame=train[feature_cols])
print(f"rows checked   {report['rows_checked']:,}")
print(f"features       {report['features_checked']}")
print(f"violations     {len(report['violations'])}\n")

if report["violations"]:
    frame = pd.DataFrame(report["violations"])
    display(frame.groupby("constraint_check_type").size().rename("count").to_frame())
    display(frame.head(12))
else:
    print("no violations: the production window looks like the training window")

unseen = report["unseen_levels"]
print(f"\nrare-bucket share in production: mean {unseen['mean_rare_rate']:.2%}, "
      f"max {unseen['max_rare_rate']:.2%} ({unseen['worst_column']})")
store.put_json(report, f"{MON}/reports/data_quality_production.json")

In [ ]:
import matplotlib.pyplot as plt

# PSI per numerical feature, and the rare-bucket share before/after, side by side.
psi_scores = {c: M.psi(train[c], production[c]) for c in C.NUM_COLS}
psi_series = pd.Series(psi_scores).sort_values(ascending=False)

rare_before = pd.Series({c: v["rare_rate"] for c, v in categorical_rates.items()})
rare_after = pd.Series({c: v["rare_rate"] for c, v in unseen["per_column"].items()})
shift = (rare_after - rare_before).sort_values(ascending=False).head(12)

fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
axes[0].bar(psi_series.index, psi_series.values, color="#6a8caf")
axes[0].axhline(M.DEFAULT_THRESHOLDS["psi_warn"], ls="--", lw=1, color="#e67e22", label="warn 0.10")
axes[0].axhline(M.DEFAULT_THRESHOLDS["psi_alarm"], ls="--", lw=1, color="#c0392b", label="alarm 0.25")
axes[0].set_title("Numerical drift (PSI), train vs production")
axes[0].tick_params(axis="x", rotation=60); axes[0].legend(fontsize=8)

axes[1].barh(shift.index[::-1], shift.values[::-1], color="#2f5d8a")
axes[1].axvline(M.DEFAULT_THRESHOLDS["unseen_rate_increase"], ls="--", lw=1,
                color="#c0392b", label="alarm: +10pp vs training")
axes[1].set_xlabel("increase in rare-bucket share")
axes[1].set_title("Unseen categorical levels: production minus training")
axes[1].legend(fontsize=8)
plt.tight_layout()
path = os.path.join(FIG_DIR, "17_data_drift.png")
plt.savefig(path, dpi=140, bbox_inches="tight")
store.upload_file(path, f"{C.PREFIX}/reports/figures/17_data_drift.png")
plt.show()

## 3 · Scheduled data-quality monitor on the endpoint

The managed monitor runs hourly against captured inference data. Two things
worth noting in the configuration:

* **`record_preprocessor_script`** is required for us. Our endpoint takes a CSV
  request carrying many rows per invocation and returns one probability per
  row, so a captured record is a *batch*, not one observation. Without the hook
  the monitor would treat five hundred scored impressions as a single record.
* **Hourly is the minimum cadence** and the first execution can take up to an
  hour to appear. That is a real constraint on the demonstration, not an error.

In [ ]:
from sagemaker.model_monitor import CronExpressionGenerator, DefaultModelMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat

code_prefix = f"{MON}/code"
preprocessor_uri = store.upload_file("../scripts/record_preprocessor.py",
                                     f"{code_prefix}/record_preprocessor.py")
print(f"record preprocessor -> {preprocessor_uri}")

data_monitor = DefaultModelMonitor(role=role, instance_count=1,
                                   instance_type="ml.m5.xlarge",
                                   volume_size_in_gb=20, max_runtime_in_seconds=1800,
                                   sagemaker_session=sess)

# Baseline through the managed container as well, so the demo can show the
# standard statistics.json / constraints.json beside ours.
data_monitor.suggest_baseline(
    baseline_dataset=s3_uri(bucket, f"{C.SPLITS_PREFIX}/xgboost/train/train.csv"),
    dataset_format=DatasetFormat.csv(header=False),
    output_s3_uri=s3_uri(bucket, f"{MON}/managed-baseline/data-quality"),
    wait=True)
print("managed data-quality baseline complete")

In [ ]:
DATA_SCHEDULE = "criteo-ctr-data-quality"
try:
    data_monitor.create_monitoring_schedule(
        monitor_schedule_name=DATA_SCHEDULE,
        endpoint_input=ENDPOINT_NAME,
        record_preprocessor_script=preprocessor_uri,
        output_s3_uri=s3_uri(bucket, f"{MON}/reports/data-quality"),
        statistics=data_monitor.baseline_statistics(),
        constraints=data_monitor.suggested_constraints(),
        schedule_cron_expression=CronExpressionGenerator.hourly(),
        enable_cloudwatch_metrics=True)
    print(f"created schedule {DATA_SCHEDULE}")
except Exception as exc:
    print(f"schedule exists or could not be created: {type(exc).__name__}: {exc}")

described = sm.describe_monitoring_schedule(MonitoringScheduleName=DATA_SCHEDULE)
print(f"status {described['MonitoringScheduleStatus']}, "
      f"cron {described['MonitoringScheduleConfig']['ScheduleConfig']['ScheduleExpression']}")

## 4 · Model quality: publish the held-back labels as ground truth

This is the step that makes model-quality monitoring possible at all. The
production labels were deliberately held back in notebook 04; they are now
written to the monitor's ground-truth prefix in the JSON-lines format it
expects, keyed by the same `inferenceId` the endpoint invocations used.

The labels are released **on a lag** rather than all at once, because that is
what real click attribution looks like: a monitoring window always contains
impressions whose outcome is not yet known. Counting those as non-clicks is the
mistake that makes a model appear to collapse at the end of every window —
`monitoring.join_predictions_to_labels` inner-joins for exactly this reason.

In [ ]:
labels = canonical_columns(store.read_parquet_prefix(f"{C.SPLITS_PREFIX}/ground_truth/"))
labels = labels[[C.RECORD_ID, C.LABEL]].sort_values(C.RECORD_ID).reset_index(drop=True)

# Release the labels for the rows the endpoint has actually scored, minus the
# most recent slice, which stands in for an open attribution window.
scored_ids = set(pd.read_parquet(
    f"s3://{bucket}/{C.PREFIX}/predictions/aligned/production_predictions.parquet"
    )[C.RECORD_ID].head(2_000))
arrived = labels[labels[C.RECORD_ID].isin(scored_ids)].head(1_400)

stamp = dt.datetime.utcnow()
gt_key = (f"{GROUND_TRUTH_PREFIX}/{ENDPOINT_NAME}/"
          f"{stamp:%Y/%m/%d/%H}/labels-{stamp:%M%S}.jsonl")
store.put_text(M.ground_truth_jsonl(arrived), gt_key)
print(f"released {len(arrived):,} of {len(scored_ids):,} scored rows as ground truth")
print(f"  -> s3://{bucket}/{gt_key}")
print(f"{len(scored_ids) - len(arrived):,} rows left unlabelled: open attribution window")

## 5 · Model-quality report from captured predictions

Read the real captured records off S3, flatten them, join to the labels that
have arrived, and compare against the training-time baseline — the same metrics
block that was attached to the registered model package, so the comparison is
against the numbers the version was approved on.

In [ ]:
capture_keys = [k for k in store.list_keys(f"{CAPTURE_PREFIX}/{ENDPOINT_NAME}/")
                if k.endswith(".jsonl")]
print(f"{len(capture_keys)} capture file(s)")

lines = []
for key in capture_keys:
    lines += s3.get_object(Bucket=bucket, Key=key)["Body"].read().decode().strip().split("\n")

captured = M.flatten_capture_records(lines)
print(f"{len(lines):,} captured invocations -> {len(captured):,} scored rows")

joined = M.join_predictions_to_labels(captured, arrived)
print(f"joined {len(joined):,}; {joined.attrs['predictions_without_labels']:,} awaiting labels")

quality = M.model_quality_report(joined, baseline_metrics)
print(f"\nstatus: {quality['status']}  ({'PASS' if quality['passed'] else 'VIOLATIONS'})")
if quality["status"] == "scored":
    display(pd.Series(quality["metrics"]).round(5).to_frame("production window"))
    display(pd.Series(quality["baseline"]).round(5).to_frame("training baseline"))
for violation in quality["violations"]:
    print(f"  - [{violation['metric_name']}] {violation['description']}")
print(f"\nretraining recommended: {quality.get('retraining_recommended')}")
store.put_json(quality, f"{MON}/reports/model_quality_window.json")

In [ ]:
# Calibration drift is the failure that costs money quietly: ranking can look
# fine while every probability is mis-scaled, and a bid is the probability
# times a value.
if quality["status"] == "scored":
    live = pd.DataFrame(quality["calibration"])
    base = pd.DataFrame(model_record["evaluation"]["evaluation_context"]["calibration"])

    fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
    axes[0].plot(base["mean_predicted"], base["observed_ctr"], "o-", color="#8aa6c1",
                 label="test split (training time)")
    axes[0].plot(live["mean_predicted"], live["observed_ctr"], "s-", color="#c0392b",
                 label="production window")
    lim = [0, max(base["mean_predicted"].max(), live["mean_predicted"].max()) * 1.15]
    axes[0].plot(lim, lim, ls="--", lw=1, color="#555", label="perfect calibration")
    axes[0].set_xlabel("mean predicted"); axes[0].set_ylabel("observed CTR")
    axes[0].set_title("Calibration: baseline vs production"); axes[0].legend(fontsize=8)

    metrics = quality["metrics"]
    names = ["log_loss", "roc_auc"]
    width = 0.35
    x = np.arange(len(names))
    axes[1].bar(x - width/2, [quality["baseline"]["log_loss"], quality["baseline"]["auc"]],
                width, label="baseline", color="#8aa6c1")
    axes[1].bar(x + width/2, [metrics["log_loss"], metrics["roc_auc"]],
                width, label="production", color="#2f5d8a")
    axes[1].set_xticks(x); axes[1].set_xticklabels(["log loss", "AUC"])
    axes[1].set_title("Model quality against the approved baseline"); axes[1].legend(fontsize=8)

    plt.tight_layout()
    path = os.path.join(FIG_DIR, "18_model_quality.png")
    plt.savefig(path, dpi=140, bbox_inches="tight")
    store.upload_file(path, f"{C.PREFIX}/reports/figures/18_model_quality.png")
    plt.show()
else:
    print(f"not scored: {quality.get('note')}")

## 6 · Scheduled model-quality monitor

Two configuration points that matter:

* We supply **our own constraints** instead of letting the monitor suggest a
  baseline. `monitoring.model_quality_constraints` derives them from the
  `evaluation.json` attached to the approved model package, so the scheduled
  job compares live windows against the exact numbers the version was approved
  on. A separately suggested baseline would drift from the registry and the two
  would disagree about what "the baseline" is.
* `problem_type="BinaryClassification"` with `probability_attribute` pointing at
  what the record preprocessor emits, so the monitor reads a score rather than
  looking for a hard class label.

In [ ]:
from sagemaker.model_monitor import EndpointInput, ModelQualityMonitor

mq_constraints = M.model_quality_constraints(baseline_metrics)
mq_key = f"{MON}/baseline/model_quality_constraints.json"
store.put_json(mq_constraints, mq_key)
print(json.dumps(mq_constraints["binary_classification_constraints"], indent=2))

quality_monitor = ModelQualityMonitor(
    role=role, instance_count=1, instance_type="ml.m5.xlarge",
    volume_size_in_gb=20, max_runtime_in_seconds=1800, sagemaker_session=sess)

MODEL_SCHEDULE = "criteo-ctr-model-quality"
endpoint_input = EndpointInput(
    endpoint_name=ENDPOINT_NAME,
    destination="/opt/ml/processing/input_data",
    inference_attribute="prediction",
    probability_attribute="prediction",
    probability_threshold_attribute=0.5)

try:
    quality_monitor.create_monitoring_schedule(
        monitor_schedule_name=MODEL_SCHEDULE,
        endpoint_input=endpoint_input,
        problem_type="BinaryClassification",
        record_preprocessor_script=preprocessor_uri,
        ground_truth_input=s3_uri(bucket, f"{GROUND_TRUTH_PREFIX}/{ENDPOINT_NAME}"),
        constraints=s3_uri(bucket, mq_key),
        output_s3_uri=s3_uri(bucket, f"{MON}/reports/model-quality"),
        schedule_cron_expression=CronExpressionGenerator.hourly(),
        enable_cloudwatch_metrics=True)
    print(f"created schedule {MODEL_SCHEDULE}")
except Exception as exc:
    print(f"schedule exists or could not be created: {type(exc).__name__}: {exc}")

for schedule in sm.list_monitoring_schedules(MaxResults=20)["MonitoringScheduleSummaries"]:
    print(f"  {schedule['MonitoringScheduleName']:34s} "
          f"{schedule['MonitoringScheduleStatus']:10s} "
          f"{schedule.get('MonitoringType', '')}")

## 7 · Clarify: feature attribution, and what it cannot tell us

Clarify answers "which features drive the predictions", which is how we detect
that the model has started relying on different signals than it did at
training. It cannot answer "is this model fair", because every categorical
feature is a **hash**: we can see that some high-cardinality identifier
dominates, and we cannot see what it identifies. That gap is stated in the
model card and the design document's security section rather than papered over
with a fairness report that does not mean what it appears to mean.

In [ ]:
from sagemaker import clarify

# The deployed model behind the endpoint is what we explain - not a fresh copy,
# so the attribution describes what is actually serving.
endpoint_config = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)["EndpointConfigName"]
serving_model = sm.describe_endpoint_config(
    EndpointConfigName=endpoint_config)["ProductionVariants"][0]["ModelName"]
print(f"explaining model {serving_model}")

explain_sample = train.head(500)[feature_cols]
explain_key = f"{MON}/clarify/explain_sample.csv"
store.put_text(explain_sample.to_csv(index=False), explain_key)

clarify_processor = clarify.SageMakerClarifyProcessor(
    role=role, instance_count=1, instance_type="ml.m5.xlarge", sagemaker_session=sess)
data_config = clarify.DataConfig(
    s3_data_input_path=s3_uri(bucket, explain_key),
    s3_output_path=s3_uri(bucket, f"{MON}/clarify/output"),
    headers=list(explain_sample.columns), dataset_type="text/csv")
model_config = clarify.ModelConfig(
    model_name=serving_model, instance_type="ml.m5.large", instance_count=1,
    accept_type="text/csv", content_type="text/csv")
shap_config = clarify.SHAPConfig(
    baseline=[explain_sample.median().tolist()],   # one median row as the reference
    num_samples=100, agg_method="mean_abs", save_local_shap_values=False)

clarify_processor.run_explainability(
    data_config=data_config, model_config=model_config,
    explainability_config=shap_config, wait=True, logs=False)
print(f"report -> {s3_uri(bucket, f'{MON}/clarify/output')}")

In [ ]:
# Global attribution, read back from the Clarify analysis output.
try:
    analysis = json.loads(s3.get_object(
        Bucket=bucket, Key=f"{MON}/clarify/output/analysis.json")["Body"].read())
    shap_values = analysis["explanations"]["kernel_shap"]["label0"]["global_shap_values"]
    top = pd.Series(shap_values).sort_values(ascending=False).head(15)[::-1]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.barh(top.index, top.values, color="#6a8caf")
    ax.set_xlabel("mean |SHAP value|")
    ax.set_title("Clarify global feature attribution (top 15)")
    plt.tight_layout()
    path = os.path.join(FIG_DIR, "19_clarify_attribution.png")
    plt.savefig(path, dpi=140, bbox_inches="tight")
    store.upload_file(path, f"{C.PREFIX}/reports/figures/19_clarify_attribution.png")
    plt.show()

    hashed = [c for c in top.index if c.startswith("C")]
    print(f"\n{len(hashed)} of the top 15 attributed features are hashed categorical "
          f"columns.\nWe can see they matter; we cannot see what they identify. That is "
          f"the\nauditability gap recorded in docs/model_card.md.")
except Exception as exc:
    print(f"analysis.json not readable yet ({type(exc).__name__}); "
          f"check {s3_uri(bucket, f'{MON}/clarify/output')}")

## 8 · Monitoring summary

What is now in place, and what each piece is actually load-bearing for:

* **Data quality**, hourly, on captured inference data — the only signal that
  arrives before the labels do. Its most informative measure for this dataset
  is the rise in unseen categorical levels, judged against the training-time
  rate per column.
* **Model quality**, hourly, joining captured predictions to released labels —
  compared against the metrics attached to the approved model package.
  Calibration error has its own threshold because calibration decays before
  ranking does, and calibration is what a bid price depends on.
* **Explainability** — feature attribution, with its limits documented rather
  than overstated.

A breached threshold does more than alert: notebook 09 wires the alarms to SNS,
and the retraining path is the same pipeline from notebook 10, with the
candidate still required to clear the quality gate and the manual approval
gate before it can replace the incumbent.

In [ ]:
summary = {
    "created_utc": pd.Timestamp.utcnow().isoformat(),
    "endpoint": ENDPOINT_NAME,
    "schedules": [s["MonitoringScheduleName"] for s in
                  sm.list_monitoring_schedules(MaxResults=20)["MonitoringScheduleSummaries"]],
    "baseline": {"statistics": s3_uri(bucket, f"{MON}/baseline/statistics.json"),
                 "constraints": s3_uri(bucket, f"{MON}/baseline/constraints.json"),
                 "training_rows": statistics["dataset"]["item_count"]},
    "data_quality": {"violations": len(report["violations"]),
                     "violation_types": sorted({v["constraint_check_type"]
                                                for v in report["violations"]}),
                     "unseen_levels": report["unseen_levels"]},
    "model_quality": quality,
    "thresholds": M.DEFAULT_THRESHOLDS,
    "ground_truth_released": int(len(arrived)),
}
store.put_json(summary, f"{C.ARTIFACTS_PREFIX}/monitoring_summary.json")
with open(os.path.abspath("../artifacts/monitoring_summary.json"), "w") as fh:
    json.dump(summary, fh, indent=2, default=str)
print(json.dumps({k: v for k, v in summary.items() if k != "model_quality"},
                 indent=2, default=str)[:1200])